In [1]:
import os
import pandas as pd
import numpy as np
import boto3
import datetime as dt
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings('ignore')

try:
    import xmltodict
except:
    ! pip install xmltodict

from parse_payload import PayloadToDataFrame

In [2]:
print(f'Latest run date: {dt.datetime.today()}')

Latest run date: 2025-02-12 15:55:23.539717


#### Constants

In [3]:
# project
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

# task
str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

# subtask
str_subtask = os.getcwd().split('/')[6]
print(f'Subtask: {str_subtask}')

# output
str_dirname_output = './output'

int_n_payloads = 10000

Project: 20241112-simple-model-test
Task: parse_all_apps
Subtask: 04_parse_payloads_dl


#### Output directory

In [4]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

#### Import data

In [5]:
str_filename = 'df_payloads.gzip'
str_uri = f's3://{str_project}/{str_task}/03_pull_data_dl/{str_filename}'
df = pd.read_parquet(str_uri)

# sort
df.sort_values(by='REQUEST_DATETIME', ascending=True, inplace=True)

# rm dup
df.drop_duplicates(subset=['ACCOUNTID'], keep='last', inplace=True)

# n rows
int_nrows = df.shape[0]

# logic
if int_n_payloads > int_nrows:
    int_n_payloads = int_nrows
else:
    pass

# get sample
df = df.sample(
    n=int_n_payloads,
    random_state=42,
)

# sort
df.sort_values(by='REQUEST_DATETIME', ascending=True, inplace=True)

# set index
df.index = list(range(0, df.shape[0]))

# show
df

,FILE_NAME,ACCOUNTID,REQUEST_DATETIME,FILE_VALUE,REQUEST_JSON,RESPONSE_JSON,REQUEST_ID,RESPONSE_MODEL_NAME,RESPONSE_MODEL_VERSION,RN
0,8425073/6dep-2024-11-26-00:01:11,8425073,2024-11-26 07:01:11+00:00,"{\n ""request"": {\n ""request_id"": ""84250737...","{\n ""request_id"": ""8425073707954"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8425073707954,PRESTIGE-GEN-XII,V1,1
1,8425080/l72Z-2024-11-26-00:03:27,8425080,2024-11-26 07:03:27+00:00,"{\n ""request"": {\n ""request_id"": ""84250809...","{\n ""request_id"": ""842508090271"",\n ""rows"": ...","{\n ""Response"": [\n {\n ""CounterOffer...",842508090271,PRESTIGE-GEN-XII,V1,1
2,8425104/c979-2024-11-26-00:09:16,8425104,2024-11-26 07:09:16+00:00,"{\n ""request"": {\n ""request_id"": ""84251042...","{\n ""request_id"": ""8425104208766"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8425104208766,PRESTIGE-GEN-XII,V1,1
3,8425136/6ga3-2024-11-26-00:18:35,8425136,2024-11-26 07:18:35+00:00,"{\n ""request"": {\n ""request_id"": ""84251367...","{\n ""request_id"": ""8425136726703"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8425136726703,PRESTIGE-GEN-XII,V1,1
4,8425173/r5q6-2024-11-26-00:29:46,8425173,2024-11-26 07:29:46+00:00,"{\n ""request"": {\n ""request_id"": ""84251738...","{\n ""request_id"": ""8425173820091"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8425173820091,PRESTIGE-GEN-XII,V1,1
...,...,...,...,...,...,...,...,...,...,...
9995,8627588/9pLH-2025-02-06-23:40:52,8627588,2025-02-07 06:40:52+00:00,"{\n ""request"": {\n ""request_id"": ""86275889...","{\n ""request_id"": ""8627588982051"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8627588982051,PRESTIGE-DLV1,V1,1
9996,8627590/97E4-2025-02-06-23:41:04,8627590,2025-02-07 06:41:04+00:00,"{\n ""request"": {\n ""request_id"": ""86275902...","{\n ""request_id"": ""8627590280509"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8627590280509,PRESTIGE-DLV1,V1,1
9997,8627613/yyYR-2025-02-06-23:47:03,8627613,2025-02-07 06:47:03+00:00,"{\n ""request"": {\n ""request_id"": ""86276137...","{\n ""request_id"": ""8627613726726"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8627613726726,PRESTIGE-DLV1,V1,1
9998,8627659/8vmh-2025-02-06-23:57:18,8627659,2025-02-07 06:57:18+00:00,"{\n ""request"": {\n ""request_id"": ""86276597...","{\n ""request_id"": ""8627659746980"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8627659746980,PRESTIGE-DLV1,V1,1


#### Iterate and parse

In [6]:
list_df_tmp = []
a = 0
for str_request in tqdm(df['REQUEST_JSON']):
    # replace NaN
    time_start = time.perf_counter()
    
    # init
    cls_parse_payload = PayloadToDataFrame()

    # get data
    df_tmp = cls_parse_payload.get_data(
        str_request=str_request,
    )

    # make copy
    df_tmp = df_tmp.copy()

    # assign
    df_tmp['ACCOUNTID'] = df['ACCOUNTID'].iloc[a]
    df_tmp['REQUEST_DATETIME'] = df['REQUEST_DATETIME'].iloc[a]
    df_tmp['RESPONSE_MODEL_NAME'] = df['RESPONSE_MODEL_NAME'].iloc[a]
    df_tmp['FILE_NAME'] = df['FILE_NAME'].iloc[a]
    
    # get number of rows
    int_nrows = df_tmp.shape[0]

    # logic
    if int_nrows == 1:
        list_bitdebtor = [1]
    else:
        list_bitdebtor = [1, 0]

    # assign
    df_tmp['BITDEBTOR'] = list_bitdebtor

    # reorder
    list_cols_id = [
        'ACCOUNTID',
        'REQUEST_DATETIME',
        'RESPONSE_MODEL_NAME',
        'FILE_NAME',
        'BITDEBTOR',
    ]
    list_cols = [col for col in df_tmp.columns if col not in list_cols_id]
    list_cols = list_cols_id + list_cols
    df_tmp = df_tmp[list_cols].copy()
    
    # end time
    time_end = time.perf_counter()
    # sec
    flt_sec = time_end - time_start

    # assign
    df_tmp['flt_sec'] = flt_sec

    # append
    list_df_tmp.append(df_tmp)
    
    # get columns
    list_cols = list(df_tmp.columns)
    
    # increase counter
    a += 1

# save memory
del df

100%|██████████| 10000/10000 [23:29<00:00,  7.10it/s]


#### Create df

In [7]:
%%time

df = pd.concat(list_df_tmp)
# save memory
del list_df_tmp
# lower
list_cols = [col.lower() for col in df.columns]
df.columns = list_cols

# show
df

CPU times: user 2min 11s, sys: 2.45 s, total: 2min 14s
Wall time: 2min 14s


,accountid,request_datetime,response_model_name,file_name,bitdebtor,uniqueid__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,dealerstate__app,...,monthlyresidual70__bb,monthlyresidual58__bb,monthlyresidual51__bb,monthlyresidual68__bb,monthlyresidual57__bb,monthlyresidual53__bb,monthlyresidual65__bb,monthlyresidual47__bb,monthlyresidual60__bb,monthlyresidual64__bb
0,8425073,2024-11-26 07:01:11+00:00,PRESTIGE-GEN-XII,8425073/6dep-2024-11-26-00:01:11,1,8.425073e+15,8425073.0,10402471.0,1,Texas,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,8425080,2024-11-26 07:03:27+00:00,PRESTIGE-GEN-XII,8425080/l72Z-2024-11-26-00:03:27,1,8.425080e+15,8425080.0,10402481.0,1,Texas,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,8425104,2024-11-26 07:09:16+00:00,PRESTIGE-GEN-XII,8425104/c979-2024-11-26-00:09:16,1,8.425104e+15,8425104.0,10402507.0,1,Texas,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,8425136,2024-11-26 07:18:35+00:00,PRESTIGE-GEN-XII,8425136/6ga3-2024-11-26-00:18:35,1,8.425136e+15,8425136.0,10402546.0,1,Texas,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,8425173,2024-11-26 07:29:46+00:00,PRESTIGE-GEN-XII,8425173/r5q6-2024-11-26-00:29:46,1,8.425173e+15,8425173.0,10402592.0,1,Texas,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,8627588,2025-02-07 06:40:52+00:00,PRESTIGE-DLV1,8627588/9pLH-2025-02-06-23:40:52,1,NaN,NaN,NaN,1,Illinois,...,6275.0,6900.0,7800.0,6475.0,7050.0,7625.0,6650.0,8150.0,6650.0,6675.0
0,8627590,2025-02-07 06:41:04+00:00,PRESTIGE-DLV1,8627590/97E4-2025-02-06-23:41:04,1,NaN,NaN,NaN,1,Louisiana,...,12200.0,13675.0,14825.0,12525.0,13875.0,14625.0,12950.0,15400.0,13375.0,13050.0
0,8627613,2025-02-07 06:47:03+00:00,PRESTIGE-DLV1,8627613/yyYR-2025-02-06-23:47:03,1,NaN,NaN,NaN,1,Texas,...,11175.0,13300.0,15225.0,11675.0,13575.0,14775.0,12250.0,15975.0,12800.0,12450.0
0,8627659,2025-02-07 06:57:18+00:00,PRESTIGE-DLV1,8627659/8vmh-2025-02-06-23:57:18,1,NaN,NaN,NaN,1,Georgia,...,10975.0,12950.0,14800.0,11425.0,13250.0,14375.0,11975.0,15525.0,12525.0,12100.0


#### Convert non-numeric to string

In [8]:
for col in tqdm(df.columns):
    # get dtype
    str_dtype = df[col].dtype
    # logic
    if str_dtype not in ['int64','float64']:
        # convert to str
        df[col] = df[col].astype(str)
    else:
        pass

100%|██████████| 2686/2686 [00:01<00:00, 2606.94it/s]


#### Write to s3

In [9]:
%%time

str_filename = 'df.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_subtask}/{str_filename}'
df.to_parquet(str_uri, compression='gzip')

CPU times: user 6.17 s, sys: 48.1 ms, total: 6.22 s
Wall time: 7.3 s
